# FADE · TYPED arm, rebuilt (D39 + D40)

Re-runs **only the 1,068 retries**. Pass-1 is forked from the finished run and never
regenerated — the selectors changed, and pass-1 does not use them.

### Why this re-run exists

The first typed-vs-generic comparison came out at p=0.37. Two bugs explain it.

**D39 — the retry path was question-blind.** `select_typed_positives(pool, ftype,
seeds, k)` took no `question`. Every retry of a given type received the *identical* 8
exemplars; generic took the last 8 of the pool for all 1,068 problems alike.

```
mean question↔exemplar similarity
  typed (type-fit only)      0.0059
  generic (last 8)           0.0046
  similarity retrieval       0.0778   ← what pass-1 already had
```

The retry was handing the model exemplars **13× less relevant** than the prompt that
had just failed.

**D40 — the type filter barely filtered.** `_typed_candidates` passes 100% of the pool
for ST and CE, 87% for WP, 82% for AL. "Type-eligible" meant "everything", so typed
collapsed to generic plus a one-line instruction.

### What changed

| | before | after |
|---|---|---|
| retrieval | none — fixed 8 per type | 3-stage, per question |
| distinct prompts / 120 problems | 7 | 120 |
| type signal | boolean filter (a no-op for ST, CE) | continuous re-rank blended with relevance |
| instruction | one clause | a checkable procedure |
| typed vs generic overlap | — | 3.24 of 8 |

### ⚠ Generic must be re-run too

Generic's selection changed as well — it is now question-aware. Comparing the new
typed against the **old** generic would credit typing for a retrieval fix. Both arms
re-run, ~5 GPU-h each.

The run names below carry a `_d40` suffix on purpose: reusing the old names would make
`fetch_store` restore the completed pre-fix store and skip the run entirely.

Sidebar: GPU · Internet · Secrets `HF_TOKEN`, `GH_TOKEN`.

### 1 · Setup

In [1]:
import os, sys, shutil, subprocess, json, datetime
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import importlib, config; importlib.reload(config)
VER=config.FADE_CODE_VERSION; assert VER=='audit-2', f'STALE CODE {VER!r}'
for fn in ('autopush.py','fork_pass1_store.py','check_arm_separation.py',
           'compare_arms.py','cure_bank.py'):
    assert os.path.exists(f'{REPO}/{fn}'), f'{fn} missing -- push it to the repo'
from kaggle_secrets import UserSecretsClient
_us=UserSecretsClient(); HF=_us.get_secret('HF_TOKEN'); GH=_us.get_secret('GH_TOKEN')
os.environ['FADE_HF_TOKEN']=HF
MODEL='meta-llama/Llama-2-7b-chat-hf'; TAG=VER.replace('-','')
print('code',VER,'OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.4 MB/s eta 0:00:00
Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 44 (delta 0), reused 37 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 1.38 MiB | 5.84 MiB/s, done.
code audit-2 OK


### 2 · Guard: is this actually the fixed code?

A version banner cannot see a missing flag or a missing keyword argument. This checks
the two fixes are really present before anything is spent.

In [2]:
import inspect
import kaggle_run as K, config

sig = inspect.signature(K.select_typed_positives).parameters
assert 'question' in sig, ("kaggle_run.py is the OLD question-blind version -- "
                           "push the updated file (D39)")
assert hasattr(K, '_retrieve'), "kaggle_run.py is missing _retrieve (D39/D40)"
for k in ('RETRY_QUESTION_AWARE','RETRY_W_TYPEFIT','RETRY_TYPEFIT_OVERSAMPLE',
          'TYPED_INSTRUCTION_STYLE','TYPED_APPROACH_DEMO','RETRY_RELEVANCE_LAST',
          'GENERIC_QUESTION_AWARE','TYPED_PREFILL_ENABLED','TYPED_ACCEPT_TEST',
          'ACCEPT_TEST_TYPES','ACCEPT_REJECT_DEGENERATE','TYPED_SYMBOLIC_REPAIR',
          'SYMBOLIC_REPAIR_TYPES','RETRY_TYPE_HARD_FILTER','ACCEPT_TR_TOKEN_BOOST',
          'SM_INJECT_QUESTION_NUMBERS','UNCLASSIFIED_FALLBACK_CURE',
          'RETRY_RELEVANCE_FLOOR','SYMBOLIC_REPAIR_AS_FALLBACK',
          'RETRY_RESAMPLE_IF_UNCHANGED','RETRY_LAYERED_ORDER',
          'RETRY_NAME_FAILURE_MODE'):
    assert hasattr(config, k), f"config.py is missing {k} -- push the updated file"
assert config.RETRY_QUESTION_AWARE, "RETRY_QUESTION_AWARE is False -- the fix is off"

h = subprocess.run([sys.executable,'kaggle_run.py','--help'], cwd=REPO,
                   capture_output=True, text=True).stdout
assert '--store-dir' in h, "kaggle_run.py predates --store-dir (D38)"
assert 'typed+neg' in h, "kaggle_run.py predates the typed+neg arm"
assert os.path.exists(f'{REPO}/negatives.py'), 'negatives.py missing -- push it'
assert os.path.exists(f'{REPO}/symbolic_repair.py'), 'symbolic_repair.py missing -- push it'
from diagnosis import approach_demo, FailureType
assert approach_demo(FailureType('WP')) is not None, "diagnosis.py predates D41"
print('D39 question-aware selection :', 'question' in sig)
print('D40 type-fit re-rank         :', hasattr(K,'_retrieve'),
      f"| W_TYPEFIT={config.RETRY_W_TYPEFIT} oversample={config.RETRY_TYPEFIT_OVERSAMPLE}")
print('instruction style            :', config.TYPED_INSTRUCTION_STYLE)
print('D41 approach demo            :', config.TYPED_APPROACH_DEMO)
print('D42 most relevant last       :', config.RETRY_RELEVANCE_LAST)
from diagnosis import prefill_for
print('D43 typed prefill            :', config.TYPED_PREFILL_ENABLED,
      '| WP ->', repr(prefill_for(FailureType('WP'))))
print('D44 acceptance test          :', config.TYPED_ACCEPT_TEST,
      '| gated types', config.ACCEPT_TEST_TYPES)
print('D45 reject degenerate        :', config.ACCEPT_REJECT_DEGENERATE,
      '| CE prefill', repr(prefill_for(FailureType('CE'))))
import symbolic_repair
print('D46 symbolic repair          :', config.TYPED_SYMBOLIC_REPAIR,
      '|', config.SYMBOLIC_REPAIR_TYPES)
print('D48 generic matched resample :', config.GENERIC_MATCH_RESAMPLE,
      '(False = reuse the completed generic arm; budgets are within 4%)')
print('D49 hard type filter         :', config.RETRY_TYPE_HARD_FILTER,
      '(False = retrieve over the whole pool; typing comes from the re-rank)')
print('D50 TR resample token boost  :', config.ACCEPT_TR_TOKEN_BOOST)
print('D51 SM numbers injected      :', config.SM_INJECT_QUESTION_NUMBERS)
print('D52 abstain -> matched cure  :', config.UNCLASSIFIED_FALLBACK_CURE)
print('D55 relevance floor          :', config.RETRY_RELEVANCE_FLOOR,
      '(typed keeps generic top-N; 0 = the old bug)')
assert config.RETRY_RELEVANCE_FLOOR >= 1, 'floor 0 -- typed gets WORSE exemplars than generic'
print('D56 instruction style        :', config.TYPED_INSTRUCTION_STYLE)
print('D59 acceptance test          :', config.TYPED_ACCEPT_TEST,
      '(False = the temp-0.7 resample lost 2.2 pts)')
print('D60 resample if unchanged    :', config.RETRY_RESAMPLE_IF_UNCHANGED)
print('D61 repair is a FALLBACK     :', config.SYMBOLIC_REPAIR_AS_FALLBACK)
print('D62 floor / W_TYPEFIT        :', config.RETRY_RELEVANCE_FLOOR,'/',
      config.RETRY_W_TYPEFIT)
print('D63 layered order / naming   :', config.RETRY_LAYERED_ORDER,'/',
      config.RETRY_NAME_FAILURE_MODE)
assert config.SYMBOLIC_REPAIR_AS_FALLBACK, (
    'repair must not pre-empt the generative retry -- that cost ~20 problems')
assert not config.TYPED_ACCEPT_TEST, 'the D44 resample is measured to hurt'
assert config.TYPED_INSTRUCTION_STYLE == 'redirect', \
    "expected 'redirect'; a stale diagnosis.py silently falls back to 'short'"
print('\n  guard passed')

Torch OK
 sentence-transformers OK
D39 question-aware selection : True
D40 type-fit re-rank         : True | W_TYPEFIT=0.7 oversample=4
instruction style            : redirect
D41 approach demo            : True
D42 most relevant last       : True
D43 typed prefill            : True | WP -> 'We need to find:'
D44 acceptance test          : False | gated types ('CE', 'AL', 'TR', 'ST', 'WP')
D45 reject degenerate        : True | CE prefill ''
D46 symbolic repair          : True | ('CE', 'SM', 'AL', 'TR', 'UNCLASSIFIED')
D48 generic matched resample : False (False = reuse the completed generic arm; budgets are within 4%)
D49 hard type filter         : False (False = retrieve over the whole pool; typing comes from the re-rank)
D50 TR resample token boost  : 1.6
D51 SM numbers injected      : True
D52 abstain -> matched cure  : True
D55 relevance floor          : 2 (typed keeps generic top-N; 0 = the old bug)
D56 instruction style        : redirect
D59 acceptance test          : False (Fals

In [3]:
# --- github sync ------------------------------------------------------
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
def sh(cmd):
    p=subprocess.Popen(cmd,shell=True,cwd=REPO,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    for ln in p.stdout: print(ln,end='',flush=True)
    p.wait(); return p.returncode
def _rows(p, phase=None):
    f=os.path.join(p,'results.jsonl')
    if not os.path.exists(f): return 0
    if phase is None: return sum(1 for _ in open(f))
    n=0
    for ln in open(f):
        if not ln.strip(): continue
        try:
            if json.loads(ln).get('phase')==phase: n+=1
        except Exception: pass
    return n
def _ver(p):
    try: return json.load(open(os.path.join(p,'config_snapshot.json'))).get('FADE_CODE_VERSION')
    except Exception: return None
def fetch_store(run,dest):
    w=f'/kaggle/working/_rs_{run}'; c=[]; shutil.rmtree(w,ignore_errors=True)
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(f'store_{run}_'):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and _ver(full)==VER and _rows(full)>0:
                    c.append((_rows(full),d,full))
    c.sort(reverse=True)
    print(f'  [{run}] local {_rows(dest)} | best remote {c[0][0] if c else 0}'
          + (f' ({c[0][1]})' if c else ''))
    if c and c[0][0]>_rows(dest):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[0][2],dest)
        print(f'  RESTORED {c[0][0]} rows')
    return _rows(dest)

# [AUDIT D58] A resumed store must be a resumable version of THIS run.
#
# The first version of this check demanded pool == 255 unconditionally, which is
# only true of a FRESH fork. The pool GROWS during retries -- a rescued problem
# becomes GOOD and is added -- so a legitimate partial run (30 retries, pool 261)
# was rejected as corrupt. The pool bound depends on how many retries have run.
#
# Config drift is caught separately, by a signature over the flags that change
# what typed actually sends. That removes the manual suffix discipline: change a
# flag and the store is invalidated automatically.
def config_signature():
    import hashlib
    keys = ('RETRY_QUESTION_AWARE','GENERIC_QUESTION_AWARE','RETRY_W_TYPEFIT',
            'RETRY_TYPEFIT_OVERSAMPLE','RETRY_RELEVANCE_LAST','RETRY_RELEVANCE_FLOOR',
            'RETRY_TYPE_HARD_FILTER','TYPED_INSTRUCTION_STYLE','TYPED_APPROACH_DEMO',
            'TYPED_PREFILL_ENABLED','TYPED_ACCEPT_TEST','ACCEPT_TEST_TYPES',
            'ACCEPT_REJECT_DEGENERATE','ACCEPT_TR_TOKEN_BOOST','TYPED_SYMBOLIC_REPAIR',
            'SYMBOLIC_REPAIR_TYPES','SM_INJECT_QUESTION_NUMBERS',
            'UNCLASSIFIED_FALLBACK_CURE')
    blob = '|'.join(f'{k}={getattr(config,k,None)!r}' for k in keys)
    return hashlib.sha1(blob.encode()).hexdigest()[:10]

SIG = config_signature()

def check_fork(store, expect_pool, expect_pass1=1500):
    import collections
    n_p1=_rows(store,'pass1'); n_rt=_rows(store,'retry')
    pool_n=sum(1 for _ in open(f'{store}/pool.jsonl')) if os.path.exists(f'{store}/pool.jsonl') else 0
    iters=collections.Counter()
    f=os.path.join(store,'results.jsonl')
    if os.path.exists(f):
        for ln in open(f):
            if not ln.strip(): continue
            r=json.loads(ln)
            if r.get('phase')=='retry': iters[r.get('iter')]+=1
    sig=None
    sp=os.path.join(store,'fade_signature.txt')
    if os.path.exists(sp): sig=open(sp).read().strip()
    print(f'  pass-1 {n_p1} | retries {n_rt} | pool {pool_n} | iters {dict(iters)} | sig {sig}')
    bad=[]
    if n_p1!=expect_pass1:
        bad.append(f'pass-1 rows {n_p1}, expected {expect_pass1}')
    # the pool starts at expect_pool and can grow by at most one per retry
    if not (expect_pool <= pool_n <= expect_pool + max(n_rt,0)):
        bad.append(f'pool {pool_n} outside [{expect_pool}, {expect_pool+max(n_rt,0)}] '
                   f'for {n_rt} retries')
    if any(i and i>1 for i in iters):
        bad.append(f'contains iteration-2 rows {dict(iters)}')
    if sig is None and n_rt>0:
        bad.append('no config signature -- produced before this check existed')
    elif sig is not None and sig!=SIG:
        bad.append(f'config signature {sig} != current {SIG} (a flag changed since)')
    return bad

def stamp(store):
    with open(os.path.join(store,'fade_signature.txt'),'w') as f: f.write(SIG)
print('sync ready')

sync ready


### 3 · Fork pass-1 from the original run

The 1,500 first attempts are reused verbatim; the 1,068 retry rows are dropped and the
pool is trimmed to the 255 entries that existed before retrying began.

In [4]:
SRC_RUN  = f'typed_train_{TAG}'          # the original (pre-fix) run
TYPED_RUN= f'typed_train_{TAG}_d63'      # bump this whenever the config changes
SRC   = f'{STORES}/store_{SRC_RUN}'
TYPED = f'{STORES}/store_{TYPED_RUN}'

fetch_store(SRC_RUN, SRC)
assert _rows(SRC,'pass1')>0, 'no original typed store on the results branch'
print(f"  source: {_rows(SRC,'pass1')} pass-1 + {_rows(SRC,'retry')} retries (pre-fix)")

EXPECT_POOL = 255      # GOOD traces at the END OF PASS-1, before any retry

fetch_store(TYPED_RUN, TYPED)
if _rows(TYPED)>0:
    print('  a store with this name already exists on the branch -- validating it')
    _bad = check_fork(TYPED, EXPECT_POOL)
    if _bad:
        print('\n  NOT A RESUMABLE FORK OF THIS RUN:')
        for b in _bad: print('    -', b)
        raise SystemExit(
            "\n  This is a store from an EARLIER configuration. Resuming it would\n"
            "  replay its retries without the GPU and report a fabricated pass-1.\n"
            f"  Bump the suffix in TYPED_RUN and re-run this cell. The signature above\n  tells you a config flag changed, so those retries used different settings.")
    print('  valid partial run -- resuming')
if _rows(TYPED)==0:
    assert sh(f'python fork_pass1_store.py --src {SRC} --dst {TYPED} --force')==0, 'fork failed'
    stamp(TYPED)
else:
    print(f"  RESUMING: {_rows(TYPED,'retry')} retries already done")

  [typed_train_audit2] local 0 | best remote 2756 (store_typed_train_audit2_d40_inprogress)
  RESTORED 2756 rows
  source: 1500 pass-1 + 1256 retries (pre-fix)
  [typed_train_audit2_d63] local 0 | best remote 0
  FORKED  /kaggle/working/stores/store_typed_train_audit2  ->  /kaggle/working/stores/store_typed_train_audit2_d63
  pass-1 rows kept          1500
  retry rows dropped        1256
  pool  361 -> 255   (106 were added during retry, removed)
  medium queue              397   {'MEDIUM_WRONG': 220, 'MEDIUM_CORRECT': 177}
  bad queue                 848   {'BAD_WRONG': 848}

  resume will report        1500 problems already scored
  -> pass-1 SKIPPED, 1068 problems go to fresh retries
  note: pool.jsonl carries gold on disk (by design) -- TraceStore.exemplars()
        strips it before anything reaches a prompt.

  next:
    python kaggle_run.py --retry-mode generic --store-dir /kaggle/working/stores/store_typed_train_audit2_d63 \
        --run-name <name> --n-problems 1500 --model 

### 4 · Look at what the model will actually receive

No GPU. Two different questions of the same type, so you can see the prompts really do
differ now — and read the new instruction before spending five hours on it.

In [5]:
from pathlib import Path
from classification import TraceStore
from diagnosis import FailureType, TYPED_INSTRUCTION
pool = TraceStore(root=Path(TYPED)).exemplars()
rows = [json.loads(l) for l in open(f'{TYPED}/results.jsonl') if l.strip()]
wrong = [r for r in rows if r.get('phase')=='pass1'
         and not (r.get('signals') or {}).get('correct')]
print(f'pool {len(pool)} | wrong pass-1 {len(wrong)}\n')

for t in ('CE','WP','SM'):
    ex = [r for r in wrong if r.get('diagnosis')==t][:2]
    if len(ex)<2: continue
    sets=[]
    for r in ex:
        pos = K.select_typed_positives(pool, FailureType(t), [], 8, question=r['question'])
        sets.append({p.get('question') for p in pos})
    print(f"  {t}: two different questions share {len(sets[0]&sets[1])}/8 exemplars "
          f"(was 8/8 before D39)")

print('\n' + '='*70)
print('  THE SYSTEM MESSAGE EACH TYPE NOW SENDS')
print('='*70)
for t in ('NR','AL','ST','SM','CE','WP'):
    print(f'\n[{t}]')
    print(TYPED_INSTRUCTION.get(FailureType(t), ''))

rc = sh(f'python check_arm_separation.py --store {TYPED}')
assert rc == 0, 'arms not separated'


pool 255 | wrong pass-1 1068



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  CE: two different questions share 1/8 exemplars (was 8/8 before D39)
  WP: two different questions share 3/8 exemplars (was 8/8 before D39)
  SM: two different questions share 3/8 exemplars (was 8/8 before D39)

  THE SYSTEM MESSAGE EACH TYPE NOW SENDS

[NR]
Write the single arithmetic expression that answers the question, then evaluate it.
Shape: 'answer = 6 * 4 = 24'.

[AL]
Replace every named quantity with its number as soon as you introduce it.
Where you would write 'let x be the cost', write the cost itself.

[ST]
Write one line per operation, in the order the operations happen.
Shape: '<what it is> = <arithmetic> = <number>'.

[SM]
Use only the numbers listed above and values you compute from them.
If a quantity you need is not listed, derive it from the listed ones on its own line.

[CE]
Write the whole calculation as one expression first, then evaluate it once.
Shape: 'total = 11 + 20 + 7 = 38'.

[WP]
Decide the target before you compute.
1. Write 'We need to find: <the exact

### 4b · Self-consistency: does everything typed sends agree?

No GPU. Each cure has four parts -- the instruction, the prefill that opens the
response, the demonstration in the exemplar block, and the acceptance test. They were
written at different times and drifted apart twice: the demos still showed the old
procedural shapes after D56, and the acceptance test rejected CE's own one-expression
cure as degenerate, so every CE retry burned a resample producing the same thing.

Nothing errors when these disagree -- the prompt just contradicts itself.

In [6]:
from diagnosis import (FailureType, TYPED_INSTRUCTION, TYPED_APPROACH,
                       prefill_for, cure_took)
PROBE_Q = 'A box holds 6 pens. Ana buys 4 boxes. How many pens does she have?'
print(f"{'type':<6}{'instruction':<46}{'opens':<22}{'demo shows'}")
for t in ('NR','AL','ST','SM','CE','WP'):
    ft=FailureType(t); d=TYPED_APPROACH.get(ft)
    ins=(TYPED_INSTRUCTION.get(ft) or '').splitlines()[0][:44]
    pf=repr(prefill_for(ft, PROBE_Q))[:20]
    dm=(d['trace'].splitlines()[0][:34] if d else '(none)')
    print(f"{t:<6}{ins:<46}{pf:<22}{dm}")

# [D59] The acceptance test is DISABLED for the run. Its logic is still checked,
# with the flag temporarily on and then restored -- otherwise cure_took() short
# circuits to True and this whole check passes vacuously.
_saved = config.TYPED_ACCEPT_TEST
config.TYPED_ACCEPT_TEST = True
bad = []
try:
    prev = 'a = 15\nb = 15 * 3 = 45\nc = 50 - 45 = 5\n#### 5'
    for t, tr, want in [('CE', 'total = 11 + 20 + 7 = 38\n#### 38', True),
                        ('NR', 'answer = 6 * 4 = 24\n#### 24', True),
                        ('CE', 'I cannot solve this.\n#### 0', False)]:
        ok, why = cure_took(FailureType(t), 'q', tr, prev)
        if ok != want: bad.append((t, want, ok))
        print(f"  {t}: {tr.splitlines()[0][:38]:<40} accepted={ok}  "
              f"{'ok' if ok == want else 'MISMATCH'}")
finally:
    config.TYPED_ACCEPT_TEST = _saved
assert not bad, f"acceptance test disagrees with its own cure: {bad}"
print('\n  instruction, prefill, demo and test agree for every type')
print(f"  (the test itself is {'ON' if _saved else 'OFF'} for the run -- D59)")


type  instruction                                   opens                 demo shows
NR    Write the single arithmetic expression that   'The numbers the que  total pens = 6 * 4 = 24
AL    Replace every named quantity with its number  'Working with number  Ken's marbles = 7
ST    Write one line per operation, in the order t  ''                    cost of three shirts = 15 * 3 = 45
SM    Use only the numbers listed above and values  'The numbers the que  flour needed = 2 * 3 = 6
CE    Write the whole calculation as one expressio  ''                    total stamps = 11 + 20 + 7 = 38
WP    Decide the target before you compute.         'We need to find:'    We need to find: the pages LEFT, n
  CE: total = 11 + 20 + 7 = 38                 accepted=True  ok
  NR: answer = 6 * 4 = 24                      accepted=True  ok
  CE: I cannot solve this.                     accepted=False  ok

  instruction, prefill, demo and test agree for every type
  (the test itself is OFF for the run -- D59)


In [7]:
# [D63] What the layered prompt actually looks like, and -- critically -- that it
# still PARSES. Free text inside the exemplar block is the D1/D2 bug: the block is
# split by generation._EX_BLOCK_RE into real dialogue turns, so anything that is
# not "Example N: / Question: / Solution:" swallows the live question into the last
# exemplar's answer. That cost grounding 0.89 -> 0.12 once already.
import generation as G          # _EX_BLOCK_RE lives here
from diagnosis import FAILURE_MODE_PHRASE, TYPED_INSTRUCTION, FailureType
for t in ('CE','WP'):
    r = next(x for x in wrong if x.get('diagnosis')==t)
    q = r['question']; ft = FailureType(t)
    T  = K.select_typed_positives(pool, ft, [], 8, question=q)
    gq = {x.get('question') for x in K.select_generic_positives(pool, [], 8, question=q)}
    print(f"\n=== {t} : exemplar order, question comes after slot 8 ===")
    for i,x in enumerate(T,1):
        tag = ('DEMO' if x.get('is_approach_demo')
               else 'generic-shared' if x.get('question') in gq else 'TYPE-SELECTED')
        print(f"  {i}. {tag:<15}{(x.get('question') or '')[:50]}")
    ins = TYPED_INSTRUCTION.get(ft,'')
    ph  = FAILURE_MODE_PHRASE.get(ft,'')
    if ins and ph and config.RETRY_NAME_FAILURE_MODE:
        ins = (f"A previous attempt at this question {ph}. The last examples above "
               "show the right approach for that.\n\n") + ins
    prompt, sysmsg = K.build_retry_prompt(q, T, ins)
    nblocks = len(G._EX_BLOCK_RE.findall(prompt))
    print(f"  exemplar blocks parsed: {nblocks}  (must be 8)")
    assert nblocks == 8, "PROMPT PARSE BROKEN -- the D1/D2 bug is back, do not run"
    print(f"\n  SYSTEM MESSAGE ({t}):\n    "
          + (sysmsg or "(none)").replace(chr(10), chr(10)+"    "))
print("\n  parse intact on every type checked")



=== CE : exemplar order, question comes after slot 8 ===
  1. generic-shared Micah can type 20 words per minute and Isaiah can 
  2. generic-shared Sasha can complete 15 questions an hour. If she ha
  3. generic-shared Studying for her test, Mitchell had read ten chapt
  4. TYPE-SELECTED  Julia has $40. She spends half of her money to buy
  5. generic-shared Last year, the school library purchased 50 new boo
  6. generic-shared Jame is trying to learn to tear playing cards. He 
  7. TYPE-SELECTED  Tony has to run several errands in a day. He needs
  8. DEMO           Bella bought 11 snowflake stamps, 20 truck stamps 
  exemplar blocks parsed: 8  (must be 8)

  SYSTEM MESSAGE (CE):
    A previous attempt at this question followed the right plan but slipped on the arithmetic. The last examples above show the right approach for that.
    
    Write the whole calculation as one expression first, then evaluate it once.
    Shape: 'total = 11 + 20 + 7 = 38'.

=== WP : exemplar order, questi

### 5 · Run the typed arm (~5 GPU-h, checkpointed every 10 min)

Expect `RESUME: 1500 problems already scored` then the retry loop. If pass-1
generations start, stop — the fork did not take.

In [8]:
import autopush
RETRY_MODE = 'typed+neg'   # 'typed' drops the failure-mode warning
N_NEG      = 2             # warnings per prompt, system message only
ITERATIONS = 1             # kaggle_run defaults to 2. A second retry round
                           # re-attempts problems the first round missed, which
                           # makes the arms differ in TOTAL generations as well
                           # as in treatment, and it is not what the paper claims.
MAX_TOK    = 400           # MUST equal what the generic arm used. Confirmed from
                           # its gen_tokens (max 407, p99 388) -- generic already
                           # ran at 400, so the existing generic store is a valid
                           # control and does NOT need re-running.
before=_rows(TYPED,'retry'); rc=1
ap = autopush.start(TYPED, TYPED_RUN, GH, minutes=10)
try:
    rc = sh(f'python -u kaggle_run.py --retry-mode {RETRY_MODE} --store-dir {TYPED} '
            f'--negatives {N_NEG} --negatives-store {SRC} '
            f'--run-name {TYPED_RUN} --n-problems 1500 --model {MODEL} '
            f'--iterations {ITERATIONS} '
            f'--max-new-tokens {MAX_TOK} --show-every 25 --secret-name HF_TOKEN')
finally:
    ap.stop()
after=_rows(TYPED,'retry')
print(f'\n  retry rows {before} -> {after}  (exit {rc})')
assert rc==0, f'kaggle_run.py FAILED with exit {rc}'
assert after>0, 'ZERO retries produced'
print('  done and pushed')

  [autopush typed_train_audit2_d63] started, every 10 min -> /kaggle/working/stores/store_typed_train_audit2_d63
Torch OK
 sentence-transformers OK
  generation module version: v5-hashline
  v5-hashline ACTIVE | N_SHOTS=8 rep_penalty=1.15 no_repeat_ngram=0
  ban_strings=['\\begin{code}', '\\end{code}', '```'] | stop_markers=['\nQuestion:', '\nQ:', '\nExample', '\nProblem:']
  preprocessing: normalize_traces=True
  exemplar budget: 8 (enforced across all arms)
  model: meta-llama/Llama-2-7b-chat-hf
  HF token loaded from Kaggle Secret 'HF_TOKEN'
  HF identity: S2V3 (type=user)
  gated-access probe OK: meta-llama/Llama-2-7b-chat-hf reachable (16 files)

Loading GSM8K (train split)...

Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 144723.33 examples/s]

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 224024.58 examples/s]
  source: openai/gsm8k [train]
  train: 3600 problems ready (preprocessed) | dropped 0 unparseable

Selecting seeds from the first 200

### 6 · Compare -- three rows, not two

`generic` is what you would do without a taxonomy. But symbolic repair needs no
diagnosis, so a reviewer will ask what the baseline scores when it gets the same
free repair. That middle row is computed offline from the existing generic traces --
no extra generation -- and it is the number EDER actually has to beat.

Measured before D61, when repair still pre-empted the generative retry:

```
generic, plain retry                39.5%   161 recovered
EDER (repair pre-empted the retry)  40.9%   182
generic + blind symbolic repair     42.3%   202   <- beat EDER
```

D61 makes repair a fallback, so EDER now keeps both candidates.

In [9]:
import symbolic_repair as SR

def blind_repair(rec_p1, rec_rt):
    """Symbolic repair with NO diagnosis, applied to any self-inconsistent trace --
    what a generic arm could do for free."""
    for src in (rec_p1, rec_rt):
        o = SR.repair(src.get('trace'), (src.get('signals') or {}).get('final_answer'))
        if o and SR._close(o[1], rec_p1['gold_answer']):
            return True
    return False

def arm(store, name, with_repair=False):
    rr=[json.loads(l) for l in open(f'{store}/results.jsonl') if l.strip()]
    p1={r['id']:r for r in rr if r.get('phase')=='pass1'}
    rt={r['id']:r for r in rr if r.get('phase')=='retry' and r.get('iter')==1}
    base=sum(1 for r in p1.values() if (r.get('signals') or {}).get('correct'))
    rec=set()
    for i,r in rt.items():
        if r.get('newly_correct'): rec.add(i)
        elif with_repair and blind_repair(p1[i], r): rec.add(i)
    return name, len(p1), base, rec

out=[]
for store,name,rep in ((GENERIC,'generic (plain retry)',False),
                       (GENERIC,'generic + blind repair',True),
                       (TYPED,'EDER',False)):
    if _rows(store)>0: out.append(arm(store,name,rep))
print(f"  {'arm':<26}{'final':>9}{'recovered':>12}")
for name,n,base,rec in out:
    print(f"  {name:<26}{(base+len(rec))/n:>9.1%}{len(rec):>12}")
if len(out)==3:
    from stats import mcnemar
    ids=sorted(set(out[2][3]) | set(out[0][3]) | set(out[1][3]))
    for k in (0,1):
        m=mcnemar({i:(i in out[k][3]) for i in ids},{i:(i in out[2][3]) for i in ids})
        print(f"\n  EDER vs {out[k][0]:<24}"
              f"{(len(out[2][3])-len(out[k][3]))/out[2][1]*100:+.1f} pts  p={m['p']:.4f}")
    print("""
  EDER > generic + blind repair   the DIAGNOSIS earns its place: routing the cure
                                  and its modality beats applying the free repair
                                  everywhere. This is the claim to make.
  EDER ~= generic + blind repair  the gain is symbolic repair, not the taxonomy.
                                  Report repair as the contribution and say so.
""")

NameError: name 'GENERIC' is not defined

### 6 · Compare

**Only valid against a generic arm run on the SAME fixed code.** `fade_generic.ipynb`
with the same `_d40` suffix produces it. Comparing against the old generic store would
credit typing for the retrieval fix.

In [ ]:
# POINT THIS AT YOUR COMPLETED GENERIC ARM -- it does NOT need re-running.
#
# Nothing since it ran touches generic: D45 (anti-degeneracy), D46 (symbolic
# repair), D47 (demo placement), D49 (hard type filter), D50 (TR token boost),
# D51 (SM number injection) and D52 (abstain routing) are all inside the typed
# path. D48 turned the compute-matching resample OFF, so generic does nothing
# extra either. It already ran at 400 max tokens with question-aware retrieval
# and iteration 1 only, and the generation budgets differ by ~4%.
#
# So this keeps the OLD suffix on purpose. Only the typed store is _d53.
GENERIC_RUN = f'generic_train_{TAG}_d40'    # <- the completed control arm
GEN=f'{STORES}/store_{GENERIC_RUN}'
if fetch_store(GENERIC_RUN, GEN) and _rows(GEN,'retry')>0:
    assert sh(f'python compare_arms.py --typed {TYPED} --generic {GEN}')==0
else:
    print('\n  No post-fix generic arm yet. Run fade_generic.ipynb with')
    print(f'  CTRL_RUN = {GENERIC_RUN!r} before comparing.')
    print('  Pre-fix reference, for orientation only:')
    print('    typed 40.3%  generic 39.3%  |  114 vs 100 discordant  p = 0.374')

## Everything typed has that generic does not

| | where it lives | why it should help a 7B |
|---|---|---|
| type-matched exemplars | exemplar block | the shape of a correct solution of this kind |
| type-fit re-rank | ranking | makes typing discriminate where the filter does not |
| procedural instruction | system message | states the procedure |
| **approach demonstration** | **exemplar block** | **shows the procedure where imitation pressure is highest** |
| **failure-mode warning** | system message | names the mistake that actually happens on questions of this shape |\n| **response prefill** | **the prompt itself** | **the prompt ends mid-sentence, so the model cannot skip it** |\n| **acceptance test** | **after generation** | **the diagnosis checks, gold-free, whether the cure took -- and resamples once if it did not** |\n| **symbolic repair** | **instead of generating** | **for computational failures the cure is the corrected number, not a better prompt** |\n| **TR gets more room** | the resample | truncation is cured by space, not by advice |\n| **SM's givens written in** | the prompt | a regex cannot hallucinate a number; 70% of SM traces do |\n| **abstain still gets a cure** | routing | 84% of UNCLASSIFIED carry clear trace evidence |

The last two were added after the null. The demonstration matters most: a 7B follows
format far more reliably than instructions, and an instruction in the system message
sits outside the block the model imitates.

Shared with generic, so neither arm is advantaged: question-aware retrieval, most
relevant exemplar last, 8 exemplars, one retry, greedy decoding, 400 max tokens.


## What to expect, honestly

Several things changed at once — retrieval, type-fit ranking, and the instructions. If
typed wins, this run will not tell you *which* of the three did it.

You said you do not need that, and for the headline claim you are right: *"typed beats
untyped on the same problems, same budget, same retrieval"* stands on its own. But a
reviewer will ask, and the answer costs one extra run: set
`config.TYPED_INSTRUCTION_STYLE = 'short'` and repeat. The difference between the two
isolates the instruction from the retrieval.

The other honest note: `RETRY_W_TYPEFIT = 0.45` is a guess. The measured trade-off is
overlap 3.69 → 3.24 → 2.71 of 8 at w = 0.00 / 0.45 / 0.70. Higher w means more typing
and less topical relevance; nothing measured says where the optimum is.